# Task C — step 4: corrected (SMT) sampler and the amortization unit test

The frozen loop with the ε-hook `−√(1−ᾱ_t)·∇_z log h_ψ`, cap by rejection (rate reported separately). Test: unweighted SMT draw vs weighted P_θ (draw C, weights from A's β*) — constraint means, held-out columns, exotics, martingale profile, sliced Wasserstein against the null SW(A, C). If they disagree the bug is in amortization, not finance; the step-count series (labelled *not P_θ* below full steps) then separates discretization from regression error. Results in `DECISIONS.md` §13.

In [ ]:
%cd /content
%rm -rf ddpm_option_pricing
!git clone https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch --all
!git checkout v2_code
!git status

In [ ]:
import os, sys, json, pickle, time
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, torch
import matplotlib.pyplot as plt
from dataclasses import replace
import taskc
from taskc.config import CFG
from taskc.ptheta import load_checkpoint, load_draw, make_schedule
from config import q_params, CONSTRAINT_LEVELS, EXOTICS     # taskb
from constraints import build                              # taskb

RUN_TAG, LEVEL = "lrema", "C3"
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
RUN = replace(CFG, artifact_dir=CFG.run_dir(RUN_TAG))
eps_model, std, _, _ = load_checkpoint(os.path.join(RUN.artifact_dir, RUN.ckpt_name), device=DEVICE)
sched = make_schedule(RUN, device=DEVICE)
tilt = pickle.load(open(os.path.join(RUN.artifact_dir, "dual", "tilts.pkl"), "rb"))[LEVEL]
q = q_params()
print("device:", DEVICE, "| P_theta:", RUN_TAG, "| level:", LEVEL)

In [ ]:
from taskc.hnet import load_hnet
from taskc.smt import sample_smt, sample_strided, sliced_wasserstein
from taskc.dual import evaluate_on, baseline_on, TASKB_EXOTICS_Q
from taskc.ptheta import save_draw
from config import HELDOUT_TESTFUNS, HELDOUT_VANILLAS
from constraints import build_heldout
from projection import wmean, wmean_se
import evaluation as ev
N_SMT, SEED = 100_000, 2001
hnet = load_hnet(os.path.join(RUN.artifact_dir, "hpsi", f"hpsi_{LEVEL}.pt"), device=DEVICE)
spec = CONSTRAINT_LEVELS[LEVEL]
A, C = load_draw(RUN, "A"), load_draw(RUN, "C")
S = sample_smt(eps_model, hnet, sched, n=N_SMT, seed=SEED, cfg=RUN, device=DEVICE, verbose=True)
out_dir = os.path.join(RUN.artifact_dir, "smt"); os.makedirs(out_dir, exist_ok=True)
save_draw(replace(RUN, artifact_dir=out_dir), f"smt_{LEVEL}", S)
print(f"SMT draw: rejected {S.n_rejected} of {S.n_drawn:,} ({S.reject_rate*100:.4f}%; P_theta draws: 0), max|z|={np.abs(S.z).max():.2f}, {S.seconds/60:.1f} min on {S.device}")

## Unit test

In [ ]:
pS, pC = std.to_paths(S.z, "SMT"), std.to_paths(C.z, "Ptheta_C")
csS, csC = build(pS, spec["testfuns"], spec["vanillas"], q), build(pC, spec["testfuns"], spec["vanillas"], q)
eC = evaluate_on(tilt, pC, q); w = eC["w"]
mS, seS = csS.G.mean(0), csS.G.std(0, ddof=1)/np.sqrt(csS.n)
mW = np.array([wmean(csC.G[:, j], w) for j in range(csC.m)]); seW = np.array([wmean_se(csC.G[:, j], w) for j in range(csC.m)])
devS, devW, dSW = (mS-csS.c)/seS, (mW-csC.c)/seW, (mS-mW)/np.hypot(seS, seW)
print(f"E[g] under SMT vs c : max {np.abs(devS).max():.2f} SE, within 2 SE {int((np.abs(devS)<=2).sum())}/{csS.m}, RMS {np.sqrt(np.mean(devS**2)):.2f}")
print(f"weighted-C vs c     : max {np.abs(devW).max():.2f} SE, within 2 SE {int((np.abs(devW)<=2).sum())}/{csC.m}, RMS {np.sqrt(np.mean(devW**2)):.2f}   (both carry the solve-draw level offset, DECISIONS 13.1)")
print(f"SMT vs weighted-C   : max {np.abs(dSW).max():.2f} SE, within 2 SE {int((np.abs(dSW)<=2).sum())}/{csS.m}, RMS {np.sqrt(np.mean(dSW**2)):.2f}   <- the sharp test")
hoS, hoC = build_heldout(pS, HELDOUT_TESTFUNS, HELDOUT_VANILLAS, q), build_heldout(pC, HELDOUT_TESTFUNS, HELDOUT_VANILLAS, q)
dh = (hoS.G.mean(0) - np.array([wmean(hoC.G[:, j], w) for j in range(hoC.m)])) / np.hypot(hoS.G.std(0, ddof=1)/np.sqrt(hoS.n), np.array([wmean_se(hoC.G[:, j], w) for j in range(hoC.m)]))
print(f"held-out columns SMT vs weighted-C: max {np.abs(dh).max():.2f} SE, within 2 SE {int((np.abs(dh)<=2).sum())}/{hoS.m}")
exS, exW = ev.price_exotics(pS), eC["exotics"]
for k in EXOTICS:
    (a, sa), (b, sb) = exS[k], exW[k]; print(f"  {k:28s} SMT {a:.4f}+-{sa:.4f}  weighted {b:.4f}+-{sb:.4f}  diff {(a-b)/np.hypot(sa,sb):+.2f} SE   Heston-Q {TASKB_EXOTICS_Q[k]:.4f}")
print(f"max |martingale deviation|: SMT {np.abs(ev.martingale_profile(pS)).max():.2e}  weighted-C {np.abs(ev.martingale_profile(pC, w)).max():.2e}  P_theta {np.abs(ev.martingale_profile(pC)).max():.2e}")
sw = dict(null=sliced_wasserstein(A.z, C.z), signal=sliced_wasserstein(C.z, C.z, wX=w), test=sliced_wasserstein(C.z, S.z, wX=w), unw_vs_smt=sliced_wasserstein(C.z, S.z))
print("sliced Wasserstein:", {k: round(v, 5) for k, v in sw.items()}, " -> test should sit at the null; unw_vs_smt at the signal")
plt.figure(figsize=(10, 3)); x = np.arange(csS.m); plt.bar(x-0.2, devS, 0.4, label="SMT vs c"); plt.bar(x+0.2, devW, 0.4, label="weighted-C vs c"); plt.axhline(2, c="k", ls="--", lw=.8); plt.axhline(-2, c="k", ls="--", lw=.8)
plt.xticks(x, csS.names, rotation=90, fontsize=6); plt.ylabel("(E[g]-c)/SE"); plt.legend(); plt.tight_layout(); plt.show()

## Step-count series (discretization vs regression) — labelled: K < 999 is **not** P_θ

In [ ]:
sd = csC.G.std(0)
print(f"{'K':>5s} {'P_theta-strided max|E[g]-c|/sd':>32s} {'SMT-strided':>12s} {'SMT RMS dev/SE':>15s}")
for K in (sched.t_start + 1, 500, 250, 100, 50):
    dP = sample_strided(eps_model, sched, 20_000, K, seed=3000+K, cfg=RUN, device=DEVICE)
    dS = sample_strided(eps_model, sched, 20_000, K, seed=4000+K, cfg=RUN, device=DEVICE, hnet=hnet)
    gP = build(std.to_paths(dP.z), spec["testfuns"], spec["vanillas"], q).G; gS = build(std.to_paths(dS.z), spec["testfuns"], spec["vanillas"], q).G
    print(f"{K:5d} {np.abs((gP.mean(0)-csS.c)/sd).max():32.4f} {np.abs((gS.mean(0)-csS.c)/sd).max():12.4f} {np.sqrt(np.mean(((gS.mean(0)-csS.c)/(gS.std(0,ddof=1)/np.sqrt(gS.shape[0])))**2)):15.2f}")
print("a plateau at the 2e4 noise level (~0.02 sd) means neither discretization nor h_psi error is visible; shrinkage with K would mean discretization.")

Stop. Results in `DECISIONS.md` §13; the cross-sample check is §13.1 (`dual/cross_sample_check.json`).